In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## データの読み込みを行う

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

oilは毎日の石油価格データ。トレーニングデータとテストデータの両方の期間における値が含まれている。エクアドルは石油依存国であり、その経済状況は石油価格の急落に強く影響を受ける。
sample_submissionはサンプル送信ファイル。
holidays_eventsはメタデータ（？）付き休日とイベント。ただし転送された列（本来7/21は海の日だけど、国の都合とかで日付がずれた場合のこと？）に注意。あと休日が追加されるパターンにも注意。
storesはcity、state、type、clusterなどのメタデータ。クラスターは類似の店舗をグループ化したもの。
trainのトレーニングデータは、store_nbr、family、onpromotionの特徴の時系列と、target salesで構成される。store_nbrは、製品が販売される店舗を識別。ファミリーは販売される製品の種類を識別。売上高は、特定の店舗における特定の日付の製品ファミリーの総売上高を示す。製品は小数単位で販売されるため、小数値も表示されます（例えば、チップス1袋ではなくチーズ1.5kg）。onpromotionは、特定の日付に店舗でプロモーションされていた製品ファミリ内のアイテムの合計数を示す。
testはトレーニングデータと同じ特徴を持つテストデータ。このファイル内の日付における目標売上を予測するのが目標。テストデータの日付は、トレーニングデータの最終日付から15日後の日付。
transactionsは日付、store_nbr、取引量で構成。

## Train.csv

In [ ]:
import pandas as pd

df_train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
display(df_train.head())

## holidays_events.csv

In [ ]:
df_holidays_events = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
display(df_holidays_events.head())

休日の日付と休日のタイプ（？）、その休日は全国なのか地方限定なのか、それが行われるのはどこなのか、休日の名前、イベントの転送がされたかどうか。

## oil.csv

In [ ]:
df_oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
display(df_oil.head())

2013年1月1日からの石油価格

## store.csv

In [ ]:
df_stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
display(df_stores.head())

店と店の場所、店がある州、店のタイプ（？）、クラスター

## transactions.csv

In [ ]:
df_transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
display(df_transactions.head())

いつ、どの店で、どれだけ取引があったか？

## test.csv

In [ ]:
df_test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
df_sample_submission = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')
display(df_test.head())
display(df_sample_submission.head())

テストデータ。2017年8月16日からの15日間（31日まで）。sample_submissionに1日あたり、1家族あたりの販売数を記入して提出するのがゴール。

## Transactiosの分析

In [ ]:
df_transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
display(df_transactions.head())

In [ ]:
transactions = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv")
import matplotlib.pyplot as plt
import seaborn as sns
transactions.head()


## transactionsとstoresを結合する

In [ ]:
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
merged_df = pd.merge(transactions, stores, on='store_nbr', how='left')
print(merged_df.head())

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='type', y='transactions', data=merged_df, estimator=sum, ci=None, palette='viridis')
plt.title('sum transactions')
plt.xlabel('store type')
plt.ylabel('transactions')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='type', y='transactions', data=merged_df, estimator=lambda x: x.mean(), ci=None, palette='viridis')
plt.title('mean transactions')
plt.xlabel('store type')
plt.ylabel('transactions')
plt.show()

## clusterとtransaction

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x='cluster', y='transactions', data=merged_df, estimator=lambda x: x.mean(), ci=None, palette='viridis')
plt.title('mean transactions')
plt.xlabel('cluster')
plt.ylabel('transactions')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
type_cluster_transactions = merged_df.groupby(['type', 'cluster'])['transactions'].mean().unstack()
print("Average number of transactions per store type and cluster:\n", type_cluster_transactions)

plt.figure(figsize=(12, 8))
sns.heatmap(type_cluster_transactions, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=.5)
plt.title('Average number of transactions per store type and cluster')
plt.xlabel('cluster')
plt.ylabel('store type')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='type', y='transactions', data=merged_df)
plt.title('Distribution of Transactions by Store Type')
plt.xlabel('store type')
plt.ylabel('transactions')
plt.ylim(0, merged_df['transactions'].quantile(0.99)) 
plt.show()

## チームメンバーが作ってくれたデータ結合コード

In [ ]:
# データ整形
# type
# 休日を１つにまとめる

import pandas as pd

# CSVファイルからデータを読み込む
holidays = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv")
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
transactions = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv")

holidays["is_holiday"] = holidays["type"].isin(["Holiday", "Additional", "Bridge"])
# イベントは個別の特徴
holidays["is_event"] = holidays["type"] == "Event"
# 休日でない場合(必要に応じて)
# holidays["is_workday"] = holidays["type"] == "Work Day"

# 表示
holidays.head()

In [ ]:
# データを結合し、mergedに格納
merged_s = pd.merge(train, stores, on="store_nbr",how="left")
merged_so = pd.merge(merged_s, oil, on="date",how="left")
merged_soh = pd.merge(merged_so, holidays, on="date",how="left")
merged = pd.merge(merged_soh, transactions, on=["date","store_nbr"],how="left")

merged.head()

In [ ]:
# 欠損値を補う
# dcoilwticoの補完、前方補完(直前のデータで補完)、直前のデータがない場合は後方補完
merged["dcoilwtico"] = merged["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
# transactions(取引数)がNaN(欠損)のときは取引数0であるため変換
merged["transactions"] = merged["transactions"].fillna(0)
# is_holiday,is_eventが欠損の場合は普通の日のためFalseに変換
merged[["is_holiday", "is_event"]] = merged[["is_holiday", "is_event"]].fillna(False)
# transferredが欠損のとき,False
merged["transferred"] = merged["transferred"].fillna(False)
# 表示
merged.head()
# date分割
# dateをdatetime型に変換
merged["date"] = pd.to_datetime(merged["date"])

# 年、月、日をそれぞれの新しい列に分ける
merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day

# 曜日（0=月曜日）
merged["dayofweek"] = merged["date"].dt.dayofweek
# 休日(土日)かどうか
merged["is_weekend"] = merged["dayofweek"] >= 5
# 周番号(必要に応じて)
# merged["weekofyear"] = merged["date"].dt.isocalendar().week

# 表示
merged.head()
# city,stateとlocale,locale_nameを用いて、店舗の地域がholiday,eventの対象か確認、対象でない場合Falseに変更
merged.loc[(merged["locale"] == "Local") & (merged["city"] != merged["locale_name"]),["is_holiday","is_event"]] = False
merged.loc[(merged["locale"] == "Regional") & (merged["state"] != merged["locale_name"]),["is_holiday","is_event"]] = False
# 表示
merged.head()

## ここまでは個人で行った作業、ここから下のデータ整理はグループで行ったもの。

In [ ]:
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
import matplotlib.pyplot as plt
import seaborn as sns
#train.head()
#oil.head()
#stores.head()
#holidays.head()

In [ ]:
#stores.head()

In [ ]:
#train['sales'].value_counts()

In [ ]:
#train.isnull().sum()

In [ ]:
#sns.stripplot(x='family', y='sales', data=train)

In [ ]:
#train['family'].value_counts()

In [ ]:
#sns.stripplot(x='store_nbr', y='sales', data=train)

In [ ]:
#sns.stripplot(x='date', y='sales', data=train)

In [ ]:
#sns.stripplot(x='onpromotion', y='sales', data=train)

In [ ]:
#sns.stripplot(x='store_nbr', y='onpromotion', data=train)

In [ ]:
#train['onpromotion'].value_counts()

In [ ]:
#train.describe()

In [ ]:
holidays["is_holiday"] = holidays["type"].isin(["Holiday", "Additional", "Bridge"])
# イベントは個別の特徴
holidays["is_event"] = holidays["type"] == "Event"
#stores.drop(['id'],axis = 1, inplace = True)
merged_s = pd.merge(train, stores, on="store_nbr",how="left")
merged_so = pd.merge(merged_s, oil, on="date",how="left")
merged_soh = pd.merge(merged_so, holidays, on="date",how="left")
merged = pd.merge(merged_soh, transactions, on=["date","store_nbr"],how="left")
merged.head()

In [ ]:
# 欠損値を補う
# dcoilwticoの補完、前方補完(直前のデータで補完)、直前のデータがない場合は後方補完
merged["dcoilwtico"] = merged["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
# transactions(取引数)がNaN(欠損)のときは取引数0であるため変換
merged["transactions"] = merged["transactions"].fillna(0)
# is_holiday,is_eventが欠損の場合は普通の日のためFalseに変換
merged[["is_holiday", "is_event"]] = merged[["is_holiday", "is_event"]].fillna(False)
# transferredが欠損のとき,False
merged["transferred"] = merged["transferred"].fillna(False)
# 表示
merged.head()
# date分割
# dateをdatetime型に変換
merged["date"] = pd.to_datetime(merged["date"])

# 年、月、日をそれぞれの新しい列に分ける
merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day

# 曜日（0=月曜日）
merged["dayofweek"] = merged["date"].dt.dayofweek
# 休日(土日)かどうか
merged["is_weekend"] = merged["dayofweek"] >= 5
# 周番号(必要に応じて)
# merged["weekofyear"] = merged["date"].dt.isocalendar().week

# 表示
merged.head()
# city,stateとlocale,locale_nameを用いて、店舗の地域がholiday,eventの対象か確認、対象でない場合Falseに変更
merged.loc[(merged["locale"] == "Local") & (merged["city"] != merged["locale_name"]),["is_holiday","is_event"]] = False
merged.loc[(merged["locale"] == "Regional") & (merged["state"] != merged["locale_name"]),["is_holiday","is_event"]] = False
# 不必要なデータ削除
merged.drop(['id','locale','locale_name','city','state','type_y','date','description'], axis=1, inplace=True)
# 表示
merged.isnull().sum()
#merged.head()

In [ ]:
training=pd.get_dummies(merged, columns=["store_nbr","family","type_x"], drop_first=True)

training

In [ ]:
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')

In [ ]:
merged_ts = pd.merge(test, stores, on="store_nbr",how="left")
merged_tso = pd.merge(merged_ts, oil, on="date",how="left")
merged_tsoh = pd.merge(merged_tso, holidays, on="date",how="left")
merged_t = pd.merge(merged_tsoh, transactions, on=["date","store_nbr"],how="left")

merged_t.head()

In [ ]:
# 欠損値を補う
# dcoilwticoの補完、前方補完(直前のデータで補完)、直前のデータがない場合は後方補完
merged_t["dcoilwtico"] = merged_t["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")
# transactions(取引数)がNaN(欠損)のときは取引数0であるため変換
merged_t["transactions"] = merged_t["transactions"].fillna(0)
# is_holiday,is_eventが欠損の場合は普通の日のためFalseに変換
merged_t[["is_holiday", "is_event"]] = merged_t[["is_holiday", "is_event"]].fillna(False)
# transferredが欠損のとき,False
merged_t["transferred"] = merged_t["transferred"].fillna(False)
# 表示
merged_t.head()

In [ ]:
# date分割
# dateをdatetime型に変換
merged_t["date"] = pd.to_datetime(merged_t["date"])

# 年、月、日をそれぞれの新しい列に分ける
merged_t["year"] = merged_t["date"].dt.year
merged_t["month"] = merged_t["date"].dt.month
merged_t["day"] = merged_t["date"].dt.day

# 曜日（0=月曜日）
merged_t["dayofweek"] = merged_t["date"].dt.dayofweek
# 休日(土日)かどうか
merged_t["is_weekend"] = merged_t["dayofweek"] >= 5
# 周番号(必要に応じて)
# merged["weekofyear"] = merged["date"].dt.isocalendar().week

# 表示
merged_t.head()

In [ ]:
# city,stateとlocale,locale_nameを用いて、店舗の地域がholiday,eventの対象か確認、対象でない場合Falseに変更
merged_t.loc[(merged_t["locale"] == "Local") & (merged_t["city"] != merged_t["locale_name"]),["is_holiday","is_event"]] = False
merged_t.loc[(merged_t["locale"] == "Regional") & (merged_t["state"] != merged_t["locale_name"]),["is_holiday","is_event"]] = False
# 表示
merged_t.head()

In [ ]:
testing=pd.get_dummies(test, columns=["store_nbr","date","family"], drop_first=True)
testing